# 4 - Machine Learning
Entrenamiento real de 11 modelos sobre ./data/titanic_procesado.csv.

In [11]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier, AdaBoostClassifier
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from sklearn.naive_bayes import GaussianNB, BernoulliNB
from sklearn.metrics import accuracy_score
import pickle
import warnings
warnings.filterwarnings('ignore')

In [12]:
# Cargar datos y verificar columnas
df = pd.read_csv('./data/titanic_procesado.csv')
print('Loaded shape:', df.shape)
X = df.drop(['Survived'], axis=1)
y = df['Survived']
print('X shape:', X.shape)
print('X columns:', list(X.columns))
assert X.shape[1] == 7, f'X debe tener 7 columnas, tiene {X.shape[1]}'

Loaded shape: (891, 8)
X shape: (891, 7)
X columns: ['Pclass', 'Sex', 'Age', 'SibSp', 'Parch', 'Fare', 'Embarked']


In [13]:
# División entrenamiento/prueba
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print('Train/test shapes:', X_train.shape, X_test.shape)

Train/test shapes: (712, 7) (179, 7)


In [14]:
# Definir modelos y grids (hiperparámetros razonables)
models = {
    'LogisticRegression': (LogisticRegression(solver='liblinear', random_state=42), {'C':[0.1,1,10]}),
    'SVC': (SVC(probability=False, random_state=42), {'C':[0.1,1,10],'kernel':['linear','rbf']}),
    'DecisionTree': (DecisionTreeClassifier(random_state=42), {'max_depth':[3,4,5]}),
    'RandomForest': (RandomForestClassifier(random_state=42), {'n_estimators':[50,100],'max_depth':[3,5]}),
    'GradientBoosting': (GradientBoostingClassifier(random_state=42), {'n_estimators':[50,100],'learning_rate':[0.05,0.1],'max_depth':[3,4]}),
    'AdaBoost': (AdaBoostClassifier(random_state=42), {'n_estimators':[50,100],'learning_rate':[0.05,0.1]}),
    'KNN': (KNeighborsClassifier(), {'n_neighbors':[3,5,7]}),
    'XGBoost': (XGBClassifier(use_label_encoder=False, eval_metric='logloss', random_state=42), {'n_estimators':[50,100],'max_depth':[3,4]}),
    'LGBM': (LGBMClassifier(random_state=42), {'n_estimators':[50,100],'max_depth':[3,4]}),
    'GaussianNB': (GaussianNB(), {}),
    'BernoulliNB': (BernoulliNB(), {'alpha':[0.1,1.0]})
}
print('Models to train:', list(models.keys()))

Models to train: ['LogisticRegression', 'SVC', 'DecisionTree', 'RandomForest', 'GradientBoosting', 'AdaBoost', 'KNN', 'XGBoost', 'LGBM', 'GaussianNB', 'BernoulliNB']


In [15]:
# Entrenamiento con GridSearchCV cuando corresponde
metricas = []
best_overall = None
best_acc = -1.0
for name, (estimator, grid) in models.items():
    print('Entrenando:', name)
    try:
        if grid:
            gs = GridSearchCV(estimator, grid, cv=5, scoring='accuracy', n_jobs=-1, verbose=0)
            gs.fit(X_train, y_train)
            best = gs.best_estimator_
        else:
            estimator.fit(X_train, y_train)
            best = estimator
        y_pred = best.predict(X_test)
        acc = accuracy_score(y_test, y_pred)
        metricas.append({'Modelo': name, 'Accuracy': acc, 'MejorEstimator': repr(best)})
        print(f'{name} accuracy: {acc:.4f}')
        if acc > best_acc:
            best_acc = acc
            best_overall = best
    except Exception as e:
        print('Error training', name, e)

metricas = pd.DataFrame(metricas).sort_values(by='Accuracy', ascending=False).reset_index(drop=True)
print('\nTabla de métricas:' )
print(metricas)
print(f'\nMejor precisión: {best_acc}')
# Guardar el mejor estimador real
if best_overall is not None:
    with open('modelo.pkl','wb') as f:
        pickle.dump(best_overall, f)
    print('Mejor estimador guardado en modelo.pkl')
else:
    print('No se entrenó ningún estimador con éxito')

Entrenando: LogisticRegression
LogisticRegression accuracy: 0.7933
Entrenando: SVC
SVC accuracy: 0.7989
Entrenando: DecisionTree
DecisionTree accuracy: 0.7989
Entrenando: RandomForest
RandomForest accuracy: 0.8156
Entrenando: GradientBoosting
GradientBoosting accuracy: 0.7989
Entrenando: AdaBoost
AdaBoost accuracy: 0.7821
Entrenando: KNN
KNN accuracy: 0.8156
Entrenando: XGBoost
XGBoost accuracy: 0.8101
Entrenando: LGBM
[LightGBM] [Info] Number of positive: 214, number of negative: 355
[LightGBM] [Info] Number of positive: 215, number of negative: 355
[LightGBM] [Info] Number of positive: 214, number of negative: 356
[LightGBM] [Info] Number of positive: 214, number of negative: 355
[LightGBM] [Info] Number of positive: 214, number of negative: 355
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.001818 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info